In [ ]:
import os
import pandas as pd
import janitor
import warnings

warnings.filterwarnings("ignore")

FP = os.path.join("..", "8-ACS_2015_19", "nhgis0002_ds244_20195_2019_county.csv")

In [ ]:
# Define and select variables to read in
census_partial = [
    "ALUBE001",  # total pop
    "ALUCE002",  # white
    "ALUCE003",  # black
    "ALULE003",  # hispanic
    "AL2OE005",  # foreign born
    "ALT0E026",  # female
    "ALW1E001",  # income
    "ALY3E005",  # unemp
    "AL2GE001",  # voting-age pop
]

# Age 29 and under
male_age29under = [
    "ALT0E003",
    "ALT0E004",
    "ALT0E005",
    "ALT0E006",
    "ALT0E007",
    "ALT0E008",
    "ALT0E009",
    "ALT0E010",
    "ALT0E011",
]
female_age29under = [
    "ALT0E027",
    "ALT0E028",
    "ALT0E029",
    "ALT0E030",
    "ALT0E031",
    "ALT0E032",
    "ALT0E033",
    "ALT0E034",
    "ALT0E035",
]
age29under = [*male_age29under, *female_age29under]

# Age 65 and over
male_age65over = [
    "ALT0E020",
    "ALT0E021",
    "ALT0E022",
    "ALT0E023",
    "ALT0E024",
    "ALT0E025",
]
female_age65over = [
    "ALT0E044",
    "ALT0E045",
    "ALT0E046",
    "ALT0E047",
    "ALT0E048",
    "ALT0E049",
]
age65over = [*male_age65over, *female_age65over]

# Education
college = ["ALWGE022", "ALWGE023", "ALWGE024", "ALWGE025"]
lesshs = [
    "ALWGE002",
    "ALWGE003",
    "ALWGE004",
    "ALWGE005",
    "ALWGE006",
    "ALWGE007",
    "ALWGE008",
    "ALWGE009",
    "ALWGE010",
    "ALWGE011",
    "ALWGE012",
    "ALWGE013",
    "ALWGE014",
    "ALWGE015",
    "ALWGE016",
]
educ = [*lesshs, *college]

identifiers = ["STATEA", "COUNTYA"]

columns_to_use = [*identifiers, *census_partial, *age29under, *age65over, *educ]

In [ ]:
# Create dict to rename columns
rename_col = {
    "ALUBE001": "population19",
    "ALUCE002": "white_pct19",
    "ALUCE003": "black_pct19",
    "ALULE003": "hispanic_pct19",
    "ALT0E026": "female_pct19",
    "ALW1E001": "median_hh_inc19",
    "ALY3E005": "clf_unemploy_pct19",
    "AL2OE005": "foreignborn_pct19",
    "AL2GE001": "votingPopulation19",
}

In [ ]:
# Read in and clean ACS 2015-19
df_acs1519 = (
    pd.read_csv(
        "nhgis0002_ds244_20195_2019_county.csv",
        usecols=columns_to_use,
        encoding="latin-1",
    )
    .rename(columns=rename_col)
    # Drop first row as description
    .pipe(lambda df: df.iloc[1:])
    .assign(
        fips=lambda df: df["STATEA"].astype(str).str.zfill(2)
        + df["COUNTYA"].astype(str).str.zfill(3)
    )
    .drop(["STATEA", "COUNTYA"], axis=1)
    # Sum age 29 and under and divide by pop
    .assign(age29under19=lambda df: df[age29under].astype("int").sum(axis=1))
    .assign(
        age29andunder_pct19=lambda df: 100
        * (df["age29under19"].astype(int) / df["population19"].astype(int))
    )
    .drop(age29under, axis=1)
    # Sum age 65 and over and divide by pop
    .assign(age65over19=lambda df: df[age65over].astype("int").sum(axis=1))
    .assign(
        age65andolder_pct19=lambda df: 100
        * (df["age65over19"].astype(int) / df["population19"].astype(int))
    )
    .drop(age65over, axis=1)
    # Sum college and divide by pop
    .assign(college_pct19=lambda df: df[college].astype("int").sum(axis=1))
    .assign(
        college_pct19=lambda df: 100
        * (df["college_pct19"].astype(int) / df["population19"].astype(int))
    )
    .drop(college, axis=1)
    # Sum lesshs and divide by pop
    .assign(lesshs_pct19=lambda df: df[lesshs].astype("int").sum(axis=1))
    .assign(
        lesshs_pct19=lambda df: 100
        * (df["lesshs_pct19"].astype(int) / df["population19"].astype(int))
    )
    .drop(lesshs, axis=1)
    # Compute other per capita census variables
    .assign(
        white_pct19=lambda df: 100
        * (df["white_pct19"].astype(int) / df["population19"].astype(int))
    )
    .assign(
        black_pct19=lambda df: 100
        * (df["black_pct19"].astype(int) / df["population19"].astype(int))
    )
    .assign(
        hispanic_pct19=lambda df: 100
        * (df["hispanic_pct19"].astype(int) / df["population19"].astype(int))
    )
    .assign(
        female_pct19=lambda df: 100
        * (df["female_pct19"].astype(int) / df["population19"].astype(int))
    )
    .assign(
        clf_unemploy_pct19=lambda df: 100
        * (df["clf_unemploy_pct19"].astype(int) / df["population19"].astype(int))
    )
    .assign(
        foreignborn_pct19=lambda df: 100
        * (df["foreignborn_pct19"].astype(int) / df["population19"].astype(int))
    )
    # Set FIPS to int
    .astype({"fips": "int"})
    .reorder_columns(["fips"])
)
df_acs1519

In [ ]:
df_acs1519.to_csv("output/acs_2015_2019.csv", index=False)